### Setting

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

import matplotlib
import matplotlib.cm as cm
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
if not hasattr(matplotlib.RcParams, '_get'):
    matplotlib.RcParams._get = matplotlib.RcParams.get

if not hasattr(matplotlib.RcParams, '_get'):
    matplotlib.RcParams._get = matplotlib.RcParams.get

import imageio.v2 as imageio
from PIL import Image, ImageDraw, ImageFont

from hex.model.tools import read_mode_config
from hex.model.framework.base_framework import baseframework

/mnt/dataset/vnwy44/miniconda3/envs/hex/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load Model

In [2]:
model_path = '../pretrained_models/EAI_real_world_2B/hex_ac100_300k_8gpu_state_query_history2/checkpoints/steps_300000_pytorch_model.pt'
infer_model = baseframework.from_pretrained(model_path)
infer_model = infer_model.to("cuda").eval()
infer_model = infer_model.to(torch.bfloat16)
infer_model.reset()
instruction = ["Follow the finger to pour the wine"]

05/17 [19:37:24] INFO     | >> [*] Loading from local checkpoint path                            ]8;id=490979;file:///mnt/dataset/vnwy44/code/HEX/hex/model/framework/share_tools.py\share_tools.py]8;;\:]8;id=931065;file:///mnt/dataset/vnwy44/code/HEX/hex/model/framework/share_tools.py#246\246]8;;\
                          `../pretrained_models/EAI_real_world_2B/hex_ac100_300k_8gpu_state_quer                   
                          y_history2/checkpoints/steps_300000_pytorch_model.pt`                                    

[EmbodimentRegistry] Saved registry to pretrained_models/hex/EAI_real_world_2B/hex_ac100_300k_8gpu_state_query_history2/embodiment_registry.json
Total number of DiT parameters:  122441736


In [3]:
class Normalizer:
    valid_modes = ["q99", "mean_std", "min_max", "binary"]

    def __init__(self, mode: str, statistics: dict):
        self.mode = mode
        self.statistics = statistics
        for key, value in self.statistics.items():
            self.statistics[key] = torch.tensor(value)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        assert isinstance(
            x, torch.Tensor
        ), f"Unexpected input type: {type(x)}. Expected type: {torch.Tensor}"

        # Normalize the tensor
        if self.mode == "q99":
            # Range of q99 is [-1, 1]
            q01 = self.statistics["q01"].to(x.dtype)
            q99 = self.statistics["q99"].to(x.dtype)

            # In the case of q01 == q99, the normalization will be undefined
            # So we set the normalized values to the original values
            mask = q01 != q99
            normalized = torch.zeros_like(x)

            # Normalize the values where q01 != q99
            # Formula: 2 * (x - q01) / (q99 - q01) - 1
            normalized[..., mask] = (x[..., mask] - q01[..., mask]) / (
                q99[..., mask] - q01[..., mask]
            )
            normalized[..., mask] = 2 * normalized[..., mask] - 1

            # Set the normalized values to the original values where q01 == q99
            normalized[..., ~mask] = x[..., ~mask].to(x.dtype)

            # Clip the normalized values to be between -1 and 1
            normalized = torch.clamp(normalized, -1, 1)

        elif self.mode == "mean_std":
            # Range of mean_std is not fixed, but can be positive or negative
            mean = self.statistics["mean"].to(x.dtype)
            std = self.statistics["std"].to(x.dtype)

            # In the case of std == 0, the normalization will be undefined
            # So we set the normalized values to the original values
            mask = std != 0
            normalized = torch.zeros_like(x)

            # Normalize the values where std != 0
            # Formula: (x - mean) / std
            normalized[..., mask] = (x[..., mask] - mean[..., mask]) / std[..., mask]

            # Set the normalized values to the original values where std == 0
            normalized[..., ~mask] = x[..., ~mask].to(x.dtype)

        elif self.mode == "min_max":
            # Range of min_max is [-1, 1]
            min = self.statistics["min"].to(x.dtype)
            max = self.statistics["max"].to(x.dtype)

            # In the case of min == max, the normalization will be undefined
            # So we set the normalized values to the original values
            mask = min != max
            normalized = torch.zeros_like(x)

            # Normalize the values where min != max
            # Formula: 2 * (x - min) / (max - min) - 1
            normalized[..., mask] = (x[..., mask] - min[..., mask]) / (
                max[..., mask] - min[..., mask]
            )
            normalized[..., mask] = 2 * normalized[..., mask] - 1

            # Set the normalized values to the original values where min == max
            # normalized[..., ~mask] = x[..., ~mask].to(x.dtype)
            # Set the normalized values to 0 where min == max
            normalized[..., ~mask] = 0

        elif self.mode == "scale":
            # Range of scale is [0, 1]
            min = self.statistics["min"].to(x.dtype)
            max = self.statistics["max"].to(x.dtype)
            abs_max = torch.max(torch.abs(min), torch.abs(max))
            mask = abs_max != 0
            normalized = torch.zeros_like(x)
            normalized[..., mask] = x[..., mask] / abs_max[..., mask]
            normalized[..., ~mask] = 0

        elif self.mode == "binary":
            # Range of binary is [0, 1]
            normalized = (x > 0.5).to(x.dtype)
        else:
            raise ValueError(f"Invalid normalization mode: {self.mode}")

        return normalized

    def inverse(self, x: torch.Tensor) -> torch.Tensor:
        assert isinstance(
            x, torch.Tensor
        ), f"Unexpected input type: {type(x)}. Expected type: {torch.Tensor}"
        if self.mode == "q99":
            q01 = self.statistics["q01"].to(x.dtype)
            q99 = self.statistics["q99"].to(x.dtype)
            return (x + 1) / 2 * (q99 - q01) + q01
        elif self.mode == "mean_std":
            mean = self.statistics["mean"].to(x.dtype)
            std = self.statistics["std"].to(x.dtype)
            return x * std + mean
        elif self.mode == "min_max":
            min = self.statistics["min"].to(x.dtype)
            max = self.statistics["max"].to(x.dtype)
            return (x + 1) / 2 * (max - min) + min
        elif self.mode == "binary":
            return (x > 0.5).to(x.dtype)
        else:
            raise ValueError(f"Invalid normalization mode: {self.mode}")

In [4]:
infer_model.norm_stats

{'tienkung2_v1': {'action': {'mean': [-0.0003965191141126146,
    0.0013129088929486243,
    0.0014843982989097028,
    -0.00022670627107733912,
    -0.0005513659235524712,
    -0.0024663440256824753,
    0.0005381562404680431,
    0.0,
    -0.004324341059628514,
    -0.001852741766454584,
    -0.0011478450097882162,
    -0.005931225893937209,
    0.00032187702889125347,
    -0.004200430855697191,
    0.0016724831860517168,
    1.1929770676860843e-05,
    0.0002980433694242384,
    0.006016429275311669,
    0.0006562193184904053,
    0.0,
    0.0,
    -0.0006659932275638077,
    0.01340800635937316],
   'std': [0.004492808686630086,
    0.01061888523299691,
    0.013028865083789625,
    0.004007827701803608,
    0.006481535165595784,
    0.020743786859589445,
    0.005394736114768051,
    0.0,
    0.04256683841874276,
    0.0193757116648689,
    0.04502628250741447,
    0.07008695478147597,
    0.016653609643585685,
    0.03829353070716905,
    0.020692347047117986,
    0.0001318023746

In [5]:
norm_stats = infer_model.norm_stats['tienkung2_v3']
action_norm_stats = norm_stats["action"]
print('action: ', action_norm_stats)
state_norm_stats = norm_stats["state"]
print('state: ', state_norm_stats)

normalizer_state = Normalizer('q99', state_norm_stats)
normalizer_action = Normalizer('q99', action_norm_stats)

action:  {'mean': [-0.012325225839650764, 0.015463482181879325, -0.0012926016203881506, -0.037432753755576635, 0.011874517689983696, -0.003303031700019405, -0.001910413312754936, 0.00020258862161810228, -0.022737016043669005, -0.013600851690754236, 0.005463684106261996, -0.032987892552929804, -0.0018960608442873696, -0.00739532735296893, 0.004107901754051708, 0.010580810445385377, -0.00034509248180850083, 0.0018331015966318478, -0.0023647193777067595, 0.026181029854634472, -0.0015048760841302618, 0.001088614745002469, 0.005862725573770673, -2.317741326140555e-07, 0.0, 0.0, 0.0, 0.030112923462986194, -0.030112923462986194, -0.030112923462986194, -0.030112923462986194, -0.030112923462986194], 'std': [0.12252177357641102, 0.10031225293869847, 0.0495085691026976, 0.22619929388386117, 0.09652154241819669, 0.036664949405365604, 0.019592201631558677, 0.0016650907383049533, 0.1622225838469734, 0.08939541011329322, 0.07655970182993421, 0.20311441673139946, 0.10328792388278582, 0.056975061349456

### Load Data

In [7]:
video_path = "./dataset/episode_000000.mp4"
save_dir = "../moe_videos/pour_wine/images"
os.makedirs(save_dir, exist_ok=True)
reader = imageio.get_reader(video_path, format="ffmpeg")
saved = 0
try:
    for i, frame in enumerate(reader):
        if i % 60 == 0:
            save_path = os.path.join(save_dir, f"frame_{i:04d}.png")
            Image.fromarray(frame).save(save_path)
            saved += 1
finally:
    reader.close()

print(f"Saved {saved} frames to {save_dir}")

Saved 29 frames to ../moe_videos/pour_wine/images


### Obtain MoE Pattern

In [6]:
df = pd.read_parquet("./dataset/episode_000000.parquet")

video_path = "./dataset/episode_000000.mp4"
save_dir = "../moe_videos"
os.makedirs(save_dir, exist_ok=True)
tags = ['tienkung2_v3']

reader = imageio.get_reader(video_path, format="ffmpeg")

normalizer_state = Normalizer("q99", state_norm_stats)
normalizer_action = Normalizer("q99", action_norm_stats)

all_action_preds = []
all_steps = []

# 如果你希望每个 step 独立预测，就 True；
# 如果希望按 episode 连续推理，只在外面 reset 一次，就 False。
reset_each_step = True

if not reset_each_step:
    infer_model.reset()

num_states = len(df)

try:
    for step, frame in tqdm(enumerate(reader), total=num_states):
        if step >= num_states:
            break

        # imageio 读出来默认是 RGB
        pil_img = Image.fromarray(frame).resize(
            (224, 224),
            Image.Resampling.LANCZOS,
        )

        batch_images = [[pil_img]]

        # state: [1, state_dim]
        state = np.stack([df["observation.state"].iloc[step]])
        state = torch.from_numpy(np.asarray(state)).float()

        # normalize state: [1, 1, state_dim]
        state = normalizer_state.forward(state).unsqueeze(0)

        if reset_each_step:
            infer_model.reset()

        with torch.no_grad():
            pred = infer_model.predict_action(
                batch_images,
                instruction,
                state,
                tags,
                return_moe_info=True
            )

        # [action_horizon, 32]
        action_pred = pred["normalized_actions"][0][:, :32]

        all_action_preds.append(action_pred)
        all_steps.append(step)

finally:
    reader.close()

all_action_preds = np.stack(all_action_preds, axis=0)
all_steps = np.asarray(all_steps)

print("all_action_preds shape:", all_action_preds.shape)
print("all_steps shape:", all_steps.shape)

np.save(os.path.join(save_dir, "episode_000000_action_preds.npy"), all_action_preds)
np.save(os.path.join(save_dir, "episode_000000_steps.npy"), all_steps)

print(f"Saved actions to {save_dir}")

/tmp/ipykernel_1842449/2938462956.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.statistics[key] = torch.tensor(value)
100%|██████████| 1722/1722 [04:07<00:00,  6.97it/s]

all_action_preds shape: (1722, 100, 32)
all_steps shape: (1722,)
Saved actions to ../moe_videos


### Visualize MoE

In [ ]:
def load_records(jsonl_path):
    records = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def extract_first_slot(record, moe_key="moe_in"):
    arr = np.array(record[moe_key]["topk_idx"])   # [1, 9, 50, 1]
    return arr[0, :, 0, 0].astype(int)            # [9]


jsonl_path = "./moe_info/moe_routing.jsonl"
save_dir = "../moe_videos/pour_wine/moe_images"
os.makedirs(save_dir, exist_ok=True)

part_names = [
    "left_arm", "right_arm", "left_hand", "right_hand",
    "left_leg", "right_leg", "head", "waist", "others"
]

moe_key = "moe_in"
# moe_key = "moe_out"

records = load_records(jsonl_path)
print(f"loaded {len(records)} frames")

selected_frames = list(range(0, len(records), 60))
if selected_frames[-1] != len(records) - 1:
    selected_frames.append(len(records) - 1)

cols = [extract_first_slot(records[i], moe_key=moe_key) for i in selected_frames]
mat = np.stack(cols, axis=1)   # [9, N]

n_cols = mat.shape[1]
fig_w = min(10, max(4.5, 0.22 * n_cols))
fig_h = 2.4

fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=300)

custom_cmap = LinearSegmentedColormap.from_list(
    "soft_blue_red",
    ["#CFE3F6", "#F7F4F2", "#F4C7C3"]
)

im = ax.imshow(
    mat,
    aspect="auto",
    interpolation="nearest",
    vmin=0,
    vmax=15,
    cmap=custom_cmap,
)

ax.set_xlabel("Input frame index", fontsize=9)
ax.set_ylabel("Body part", fontsize=9)

ax.set_yticks(np.arange(len(part_names)))
ax.set_yticklabels(part_names, fontsize=8)

show_every = max(1, n_cols // 8)
xticks = np.arange(0, n_cols, show_every)
if xticks[-1] != n_cols - 1:
    xticks = np.append(xticks, n_cols - 1)

ax.set_xticks(xticks)
ax.set_xticklabels([str(selected_frames[i]) for i in xticks], fontsize=7)

ax.set_xticks(np.arange(-0.5, n_cols, 1), minor=True)
ax.set_yticks(np.arange(-0.5, mat.shape[0], 1), minor=True)
ax.grid(which="minor", color="white", linestyle="-", linewidth=0.35, alpha=0.5)
ax.tick_params(which="minor", bottom=False, left=False)

for spine in ax.spines.values():
    spine.set_linewidth(0.6)

cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02, ticks=[0, 3, 6, 9, 12, 15])
cbar.set_label("Expert ID", fontsize=8)
cbar.ax.tick_params(labelsize=7)

fig.tight_layout()

save_path = os.path.join(save_dir, f"{moe_key}_slot0_compact.png")
plt.savefig(save_path, bbox_inches="tight", pad_inches=0.02, transparent=True,)
plt.close(fig)

print(f"saved to {save_path}")

loaded 1725 frames
saved to ../moe_videos/pour_wine/moe_images/moe_in_slot0_compact_gradient.png


In [ ]:
jsonl_path = "./moe_info/moe_routing.jsonl"
save_dir = "../moe_videos/pour_wine/moe_images"
os.makedirs(save_dir, exist_ok=True)

part_names = [
    "left_arm", "right_arm", "left_hand", "right_hand",
    "left_leg", "right_leg", "head", "waist", "others"
]

def load_records(jsonl_path):
    records = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

records = load_records(jsonl_path)
print(f"loaded {len(records)} frames")

def extract_grid(record, moe_key="moe_in"):
    arr = np.array(record[moe_key]["topk_idx"])
    arr = arr[0, :, :, 0]  # [1, 9, 50, 1] -> [9, 50]
    return arr.astype(int)

# generate expert id -> color
cmap = cm.get_cmap("viridis", 16)
color_lut = []
for i in range(16):
    r, g, b, _ = cmap(i)
    color_lut.append((int(r * 255), int(g * 255), int(b * 255)))

def render_frame_fast(grid, frame_idx, moe_key="moe_in"):
    H, W = grid.shape

    cell_w = 24
    cell_h = 32
    left_margin = 150
    top_margin = 55
    bottom_margin = 35
    right_margin = 20

    img_w = left_margin + W * cell_w + right_margin
    img_h = top_margin + H * cell_h + bottom_margin

    img = Image.new("RGB", (img_w, img_h), "white")
    draw = ImageDraw.Draw(img)

    try:
        font_title = ImageFont.truetype("DejaVuSans.ttf", 18)
        font_label = ImageFont.truetype("DejaVuSans.ttf", 13)
        font_cell = ImageFont.truetype("DejaVuSans.ttf", 10)
    except:
        font_title = ImageFont.load_default()
        font_label = ImageFont.load_default()
        font_cell = ImageFont.load_default()

    # title
    draw.text(
        (left_margin, 15),
        f"{moe_key} | input frame {frame_idx}",
        fill="black",
        font=font_title,
    )

    # x labels every 5
    for j in range(0, W, 5):
        x = left_margin + j * cell_w + cell_w // 2
        draw.text((x - 6, top_margin - 22), str(j), fill="black", font=font_label)

    # body part labels
    for i, name in enumerate(part_names):
        y = top_margin + i * cell_h + cell_h // 2 - 7
        draw.text((8, y), name, fill="black", font=font_label)

    # cells
    for i in range(H):
        for j in range(W):
            val = int(grid[i, j])
            color = color_lut[val % 16]

            x0 = left_margin + j * cell_w
            y0 = top_margin + i * cell_h
            x1 = x0 + cell_w
            y1 = y0 + cell_h

            draw.rectangle([x0, y0, x1, y1], fill=color, outline="white")

            text_color = "white" if val >= 8 else "black"
            text = str(val)

            # 简单居中
            bbox = draw.textbbox((0, 0), text, font=font_cell)
            tw = bbox[2] - bbox[0]
            th = bbox[3] - bbox[1]

            draw.text(
                (x0 + (cell_w - tw) / 2, y0 + (cell_h - th) / 2 - 1),
                text,
                fill=text_color,
                font=font_cell,
            )

    return np.asarray(img)

def save_video_fast(records, moe_key="moe_in", fps=30):
    out_path = os.path.join(save_dir, f"{moe_key}_numeric.mp4")

    writer = imageio.get_writer(
        out_path,
        fps=fps,
        codec="libx264",
        quality=8,
        macro_block_size=1,
    )

    for frame_idx, record in tqdm(enumerate(records), total=len(records), desc=moe_key):
        grid = extract_grid(record, moe_key=moe_key)
        img = render_frame_fast(grid, frame_idx, moe_key=moe_key)
        writer.append_data(img)

    writer.close()
    print(f"saved to {out_path}")
    return out_path

moe_in_video = save_video_fast(records, moe_key="moe_in", fps=30)
moe_out_video = save_video_fast(records, moe_key="moe_out", fps=30)

print("done")
print(moe_in_video)
print(moe_out_video)

/tmp/ipykernel_1842449/1836440327.py:36: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap("viridis", 16)


loaded 1725 frames


moe_in: 100%|██████████| 1725/1725 [01:24<00:00, 20.45it/s]


saved to ../moe_videos/pour_wine/moe_images/moe_in_numeric_fast.mp4


moe_out: 100%|██████████| 1725/1725 [01:23<00:00, 20.56it/s]


saved to ../moe_videos/pour_wine/moe_images/moe_out_numeric_fast.mp4
done
../moe_videos/pour_wine/moe_images/moe_in_numeric_fast.mp4
../moe_videos/pour_wine/moe_images/moe_out_numeric_fast.mp4
